# Signature Verification — Siamese CNN

The plain CNN couldn't really learn *similarity*. A Siamese network fixes that: the same CNN tower
(shared weights) turns each signature into an embedding vector, and we compare the two embeddings
by Euclidean distance.

We train with **contrastive loss** so genuine pairs end up close together and forged pairs end up
far apart. At test time we just pick a distance threshold.

**Techniques used:** shared-weight towers, Conv2D + BatchNorm + Dropout, Lambda distance layer, contrastive loss, EER threshold

## 1. Imports

In [ ]:
import os, json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from keras.models import Sequential, Model
from keras.layers import Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Lambda
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, roc_curve

## 2. Get the Dataset

In [ ]:
# clone the project repo (run once) — the dataset ships inside it
!git clone https://github.com/goyashek/Signature-forgery-verification.git

DATA_ROOT = 'Signature-forgery-verification/sign_data'
IMG_DIR   = os.path.join(DATA_ROOT, 'train')

IMG_H, IMG_W = 150, 220

## 3. Writer-independent split + leak-free pairing

Two data problems to avoid here (both diagnosed in `01b_data_leak_investigation.ipynb`):

1. **Duplicate test folder** — the shipped `test/` is a copy of `train/`, so we split by **writer
   id** and test only on unseen people (train ≤40, val 41–48, test ≥49).
2. **The pairing leak** — the shipped CSV only contains *genuine-vs-genuine* (match) and
   *genuine-vs-forgery* (non-match) pairs. That makes the label a giveaway of which folder `img2`
   came from, so a model can "cheat" by only looking at the questioned image. We rebuild the pairs
   from the raw per-writer folders and add a **third recipe — genuine vs a *different writer's*
   genuine** — as an extra non-match. Now a genuine `img2` can be either label, so the model is
   forced to actually compare the two signatures.

In [ ]:
# Index the raw folders: genuine[w] = list of w's genuine files, forg[w] = w's forgeries.
# (We build pairs ourselves instead of reading the CSV, because the CSV lacks the
#  different-writer negatives we need to kill the leak.)
import random
from collections import defaultdict

genuine, forg = defaultdict(list), defaultdict(list)
for d in sorted(os.listdir(IMG_DIR)):
    full = os.path.join(IMG_DIR, d)
    if not os.path.isdir(full):
        continue
    files = [os.path.join(d, f) for f in os.listdir(full) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if d.endswith('_forg'):
        forg[int(d.replace('_forg', ''))].extend(files)
    else:
        genuine[int(d)].extend(files)

writers = sorted(set(genuine) & set(forg))
train_writers = [w for w in writers if w <= 40]
val_writers   = [w for w in writers if 41 <= w <= 48]
test_writers  = [w for w in writers if w >= 49]
print('writers ->', len(train_writers), 'train |', len(val_writers), 'val |', len(test_writers), 'test')

## 4. Build & load the pairs

Label semantics: **0 = same writer (similar), 1 = non-match (different)**. A non-match is now either
a *forgery* of the writer or a *different writer's genuine* signature. We keep the two signatures as
two separate arrays, since each goes through the tower on its own.

In [ ]:
def make_pairs(wset, per_writer, seed=42):
    """Build balanced pairs from the folders. Three recipes:
       match (0): two genuine of the same writer
       hard negative (1): genuine vs a forgery of that writer
       random negative (1): genuine vs a *different* writer's genuine  <- kills the leak
    """
    rng = random.Random(seed)
    wlist = sorted(wset)
    rows = []
    for w in wlist:
        g = genuine.get(w, [])
        if len(g) < 2:
            continue
        for _ in range(per_writer):                       # match
            a, b = rng.sample(g, 2)
            rows.append((a, b, 0))
        for _ in range(per_writer // 2):                  # hard negative: forgery of w
            if forg.get(w):
                rows.append((rng.choice(g), rng.choice(forg[w]), 1))
        for _ in range(per_writer // 2):                  # random negative: different writer
            other = rng.choice([x for x in wlist if x != w])
            rows.append((rng.choice(g), rng.choice(genuine[other]), 1))
    rng.shuffle(rows)
    return pd.DataFrame(rows, columns=['img1', 'img2', 'label'])

def load_pairs(frame):
    X1, X2, y = [], [], []
    for _, row in frame.iterrows():
        a = cv2.imread(os.path.join(IMG_DIR, row['img1']), cv2.IMREAD_GRAYSCALE)
        b = cv2.imread(os.path.join(IMG_DIR, row['img2']), cv2.IMREAD_GRAYSCALE)
        a = cv2.resize(a, (IMG_W, IMG_H))
        b = cv2.resize(b, (IMG_W, IMG_H))
        X1.append(a); X2.append(b); y.append(row['label'])
    X1 = np.array(X1, dtype='float32')[..., None] / 255.0
    X2 = np.array(X2, dtype='float32')[..., None] / 255.0
    return X1, X2, np.array(y)

# per_writer controls dataset size: ~2*per_writer pairs per writer (1 match + 0.5 + 0.5 negatives)
X1_tr, X2_tr, y_tr = load_pairs(make_pairs(train_writers, per_writer=100))
X1_va, X2_va, y_va = load_pairs(make_pairs(val_writers,   per_writer=100, seed=1))
X1_te, X2_te, y_te = load_pairs(make_pairs(test_writers,  per_writer=100, seed=2))
print('train:', X1_tr.shape, '| val:', X1_va.shape, '| test:', X1_te.shape)
print('train label balance:', dict(zip(*np.unique(y_tr, return_counts=True))))

## 5. The shared tower

One small CNN that maps a signature to a 128-d embedding. Because we reuse this same `tower` model
for both inputs, the weights are shared automatically.

In [ ]:
tower = Sequential([
    Input(shape=(IMG_H, IMG_W, 1)),
    Conv2D(32, (3, 3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(),
    Conv2D(64, (3, 3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(),
    Conv2D(128, (3, 3), activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(),
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128),
], name='tower')
tower.summary()

## 6. Assemble the Siamese model

Two inputs go through the same tower, then a Lambda layer computes the Euclidean distance between
the two embeddings.

In [ ]:
def euclidean_distance(vecs):
    a, b = vecs
    return tf.sqrt(tf.reduce_sum(tf.square(a - b), axis=1, keepdims=True) + 1e-9)

input_a = Input(shape=(IMG_H, IMG_W, 1))
input_b = Input(shape=(IMG_H, IMG_W, 1))

emb_a = tower(input_a)
emb_b = tower(input_b)
distance = Lambda(euclidean_distance)([emb_a, emb_b])

model = Model([input_a, input_b], distance)

## 7. Contrastive loss & training

Genuine pairs (y=0) are pulled together; forged pairs (y=1) are pushed apart up to a margin.

In [ ]:
def contrastive_loss(y_true, d):
    margin = 1.0
    y_true = tf.cast(y_true, tf.float32)
    return tf.reduce_mean((1 - y_true) * tf.square(d) +
                          y_true * tf.square(tf.maximum(margin - d, 0)))

model.compile(optimizer=Adam(1e-3), loss=contrastive_loss)

early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

history = model.fit([X1_tr, X2_tr], y_tr,
                    validation_data=([X1_va, X2_va], y_va),
                    epochs=30, batch_size=32,
                    callbacks=[early_stop, reduce_lr])

In [ ]:
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Contrastive loss'); plt.legend(); plt.show()

## 8. Pick a threshold

The model gives a distance, not a yes/no. We pick the cutoff on the validation set at the Equal
Error Rate (where false accepts = false rejects). Distance below the cutoff -> genuine.

In [ ]:
val_d = model.predict([X1_va, X2_va]).ravel()

# EER: find the threshold where false-accept rate ~ false-reject rate
fpr, tpr, thr = roc_curve(y_va, val_d)
fnr = 1 - tpr
eer_idx = np.nanargmin(np.abs(fpr - fnr))
threshold = thr[eer_idx]
print('threshold:', round(float(threshold), 4))
print('val EER:  ', round(float((fpr[eer_idx] + fnr[eer_idx]) / 2) * 100, 2), '%')

## 9. Evaluate on unseen writers

In [ ]:
test_d = model.predict([X1_te, X2_te]).ravel()
pred = (test_d > threshold).astype(int)   # far -> forged

acc = (pred == y_te).mean()
auc = roc_auc_score(y_te, test_d)
far = (test_d[y_te == 1] < threshold).mean()   # forgery accepted as genuine
frr = (test_d[y_te == 0] > threshold).mean()   # genuine rejected as forged

print('Accuracy:', round(acc * 100, 2), '%')
print('ROC-AUC :', round(auc, 3))
print('FAR     :', round(far * 100, 2), '%')
print('FRR     :', round(frr * 100, 2), '%')

In [ ]:
# how well do the two distance distributions separate?
plt.hist(test_d[y_te == 0], bins=40, alpha=0.6, label='genuine')
plt.hist(test_d[y_te == 1], bins=40, alpha=0.6, label='forged')
plt.axvline(threshold, color='k', ls='--', label='threshold')
plt.xlabel('distance'); plt.legend(); plt.show()

## 10. Save the tower + threshold

We save the tower (not the full pair-model) so the Streamlit app can embed one signature at a time.

In [ ]:
tower.save('siamese_embedding.keras')
json.dump({'threshold': float(threshold), 'img_h': IMG_H, 'img_w': IMG_W, 'model': 'siamese_cnn'},
          open('siamese_cnn_meta.json', 'w'))
print('saved tower + meta')

## 11. Takeaways

- With leak-free pairs, the Siamese network has to learn an actual distance metric — it can't fall
  back on the "is `img2` forged?" shortcut that inflated notebook 1. Genuine and forged pairs land
  in different parts of the embedding space, and it holds up on writers it never trained on.
- We can enroll a brand new person from a single reference signature, no retraining needed.
- The tower is still trained from scratch on fairly little data, so expect the honest numbers to be
  lower than notebook 1's leaky 0.999 — that's the point. These are numbers you can trust.
- **Next:** swap the from-scratch tower for a pretrained MobileNetV2 (transfer learning) and see
  if borrowed ImageNet features push the numbers up.